# Capstone Track A — Domain RAG Assistant
**Day 2 Afternoon | ~3 hours | Colab CPU | `OPENAI_API_KEY`**

---

You have built every piece of this already. Lab 6 was the pipeline, Lab 7 was the app. What is new is the domain: your documents, your users, your questions. Most of what makes a RAG assistant good or bad lives in choices the labs made for you, and here you make them yourself.

**Your decisions, marked `TODO` in the cells:**

- [ ] Step 1: Choose a domain and load real documents
- [ ] Step 2: Choose a chunk size, and write down why
- [ ] Step 3: Write the grounding prompt for your domain
- [ ] Step 4: Test retrieval on questions you picked, including one it should refuse
- [ ] Step 5: Give the app a name and three example questions
- [ ] Step 6: Score it, and find one question it gets wrong

The plumbing (embedding, the vector store, streaming, the Gradio shell, RAGAS) is provided and works as-is. Spend your time on the decisions.

> **Track B (fine-tuning)?** Open `capstone_track_b.ipynb` instead. It needs a T4 GPU.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} sentence-transformers chromadb "langchain-community<0.4" langchain-core langchain-text-splitters langchain-openai openai "gradio>=6" pypdf beautifulsoup4 ragas python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-4o-mini"        # generator
JUDGE_MODEL     = "gpt-4o"             # RAGAS judge (Step 6)
EMBED_MODEL     = "all-MiniLM-L6-v2"   # local embeddings, no key
print(f"Ready — generate with {DEFAULT_MODEL}, embed with {EMBED_MODEL}")

---

## Step 1 — Load your documents

Pick **one** option and run it. Each one ends with the same thing: a list called `documents`, where every item is a LangChain `Document` with a `source` in its metadata. Everything after this step only needs that list.

| Option | Good for | Effort |
|--------|----------|--------|
| **A — Inline text** | Getting the pipeline working first | Low |
| **B — Upload PDFs** | Documents you have on your laptop | Medium |
| **C — Web pages** | Public docs, Wikipedia, a project's guide | Low |

Start with A, run the notebook top to bottom once so you know the plumbing works, then come back and switch to your real documents.

A few pages of good, focused text beats a pile of loosely related material. Lab 6 showed what happens when one big source crowds out the rest of the index.

In [ ]:
# Option A — inline text. Always works, no upload.
from langchain_core.documents import Document

# TODO: replace with your domain's content. Aim for at least 3 topics, a solid paragraph each.
raw_documents = {
    "topic_1": '''
    Paste your first document or topic summary here.
    Make it at least a paragraph. Short snippets retrieve poorly.
    ''',
    "topic_2": '''
    Paste your second document here.
    ''',
    "topic_3": '''
    Paste your third document here.
    ''',
}

documents = [Document(page_content=text.strip(), metadata={"source": name})
             for name, text in raw_documents.items()]
print(f"Loaded {len(documents)} documents (inline)")

In [ ]:
# Option B — upload PDFs from your computer. Uncomment to use; a file picker opens in Colab.

# from google.colab import files
# from langchain_community.document_loaders import PyPDFLoader
#
# uploaded = files.upload()                       # choose one or more PDFs
# documents = []
# for fname in uploaded:
#     pages = PyPDFLoader(fname).load()           # one Document per page, source = file name
#     documents.extend(pages)
#     print(f"{fname}: {len(pages)} pages")
# print(f"Loaded {len(documents)} pages from {len(uploaded)} PDFs")

In [ ]:
# Option C — web pages. Uncomment and put your own URLs in.

# from langchain_community.document_loaders import WebBaseLoader
#
# urls = [                                        # TODO: URLs from your domain
#     "https://en.wikipedia.org/wiki/YOUR_TOPIC_1",
#     "https://en.wikipedia.org/wiki/YOUR_TOPIC_2",
# ]
# documents = WebBaseLoader(urls).load()          # source = the URL
# print(f"Loaded {len(documents)} pages")

---

## Step 2 — Chunk and index

**Your decision: the chunk size.**

- **Smaller (200 to 400 characters):** sharper matches, but an answer that spans a few sentences may be split across chunks.
- **Larger (800 to 1,200 characters):** more context per chunk, but more text in it that has nothing to do with the question.

Dense reference material (specs, FAQs, API docs) usually wants smaller chunks. Narrative text (reports, policies, articles) usually wants larger ones. Pick a size, say why in the comment, and come back here if Step 4 shows retrieval pulling the wrong chunks.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: choose your chunk size and justify it
# My documents are [X], so I chose [Y] characters because [Z].
CHUNK_SIZE    = 400
CHUNK_OVERLAP = 80     # about 20% of the chunk size

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunks = splitter.split_documents(documents)

print(f"{len(chunks)} chunks from {len(documents)} documents")
print("First chunk:", chunks[0].page_content[:150])

In [ ]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

collection = chromadb.Client().get_or_create_collection(
    "capstone",
    embedding_function=SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL),
    configuration={"hnsw": {"space": "cosine"}},
)
existing = collection.get()["ids"]              # re-run safe
if existing:
    collection.delete(ids=existing)

collection.add(
    documents=[c.page_content for c in chunks],
    metadatas=[{"source": str(c.metadata.get("source", "unknown"))} for c in chunks],
    ids=[f"c{i}" for i in range(len(chunks))],
)
print(f"Indexed {collection.count()} chunks")

---

## Step 3 — Write your grounding prompt

This is where most of your assistant's behaviour is decided: whether it sticks to your documents, how it sounds, and what it does when the documents do not have the answer.

`"You are a helpful assistant."` gives you none of that. A useful prompt says who the assistant is for, holds it to the context, tells it how to cite, and tells it what to do when the context falls short.

Lab 6 found one trap worth avoiding. A prompt that says only *answer ONLY from the context, otherwise say you don't know* makes `gpt-4o-mini` refuse questions it has the context for, because refusing is the one safe move it was given. Allow a partial answer that names what is missing, and keep the full refusal for questions the context does not touch.

Keep the `{context}` placeholder: that is where the retrieved chunks go.

In [ ]:
from openai import OpenAI

oai = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

# TODO: write your domain-specific system prompt
SYSTEM_PROMPT = """You are an assistant for [WHO YOUR USERS ARE], answering questions about [YOUR DOMAIN].
Answer using only the context below. Cite the [source] of each fact you use.
If the context answers part of the question, answer that part and say what is missing.
If it answers none of it, say "I don't have information about that in my documents."

Context:
{context}"""

In [ ]:
def search(query, n=3):
    res = collection.query(query_texts=[query], n_results=n)
    return res["documents"][0], res["metadatas"][0]

def build_messages(question):
    docs, metas = search(question)
    context = "\n\n".join(f"[{m['source']}]: {d}" for d, m in zip(docs, metas))
    messages = [{"role": "system", "content": SYSTEM_PROMPT.format(context=context)},
                {"role": "user", "content": question}]
    return messages, docs, metas

def rag(question):
    messages, docs, metas = build_messages(question)
    answer = oai.chat.completions.create(model=DEFAULT_MODEL, messages=messages).choices[0].message.content
    return answer, docs, metas

print("search, build_messages and rag are ready")

---

## Step 4 — Test retrieval before you build the UI

Check three things before any interface exists. If one of them fails here, it will fail in the app too, and it is much easier to see here.

1. A question your documents **do** answer. Read the chunks first: is the answer actually in them? Then read the answer: did it stay inside them?
2. A question your documents **do not** answer. It should say so, not improvise.
3. If Step 1 used several sources, a question that needs one specific source. Did the right one come back?

When the answer is wrong, look at the retrieved chunks before you touch the prompt. Wrong chunks are a Step 2 problem (chunk size) or a Step 1 problem (documents).

In [ ]:
# Test 1 — a question your documents answer
q1 = "YOUR IN-SCOPE QUESTION HERE"          # TODO

answer, docs, metas = rag(q1)
for m, d in zip(metas, docs):
    print(f"[{m['source']}] {d[:100]}...")
print("\nAnswer:", answer)

In [ ]:
# Test 2 — a question your documents do NOT answer (it should say so)
q2 = "YOUR OUT-OF-SCOPE QUESTION HERE"      # TODO

answer, docs, metas = rag(q2)
print("Sources:", [m["source"] for m in metas])
print("Answer :", answer)

---

## Step 5 — Build the app

This is Lab 7's app, trimmed. You own the name, the description and the three example questions. The callback below streams the answer, fills the sources panel and logs the latency. Run the next two cells and you get a public link.

In [ ]:
import time
import gradio as gr

# TODO: make it yours
APP_TITLE         = "My Domain Assistant"
APP_DESCRIPTION   = "Ask me about [YOUR DOMAIN]."
EXAMPLE_QUESTIONS = [
    "YOUR EXAMPLE QUESTION 1",
    "YOUR EXAMPLE QUESTION 2",
    "YOUR EXAMPLE QUESTION 3",
]

query_log = []

def respond(message, chat_history):
    t0 = time.time()
    messages, docs, metas = build_messages(message)
    sources_md = "\n\n".join(f"**{m['source']}**: _{d[:120]}..._" for d, m in zip(docs, metas))

    chat_history = chat_history + [{"role": "user", "content": message},
                                   {"role": "assistant", "content": ""}]
    answer = ""
    for chunk in oai.chat.completions.create(model=DEFAULT_MODEL, messages=messages, stream=True):
        answer += chunk.choices[0].delta.content or ""
        chat_history[-1] = {"role": "assistant", "content": answer}
        yield "", chat_history, sources_md

    latency_ms = int((time.time() - t0) * 1000)
    query_log.append({"question": message, "latency_ms": latency_ms})
    yield "", chat_history, sources_md + f"\n\n_{latency_ms} ms_"

In [ ]:
with gr.Blocks(title=APP_TITLE) as demo:
    gr.Markdown(f"# {APP_TITLE}\n{APP_DESCRIPTION}")
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=420)
            msg     = gr.Textbox(placeholder="Ask a question...", show_label=False)
            gr.Examples(EXAMPLE_QUESTIONS, inputs=msg)
        with gr.Column(scale=2):
            sources_box = gr.Markdown("*Retrieved sources appear here.*")

    msg.submit(respond, [msg, chatbot], [msg, chatbot, sources_box])

# The public link lives as long as this runtime. demo.close() stops it.
demo.launch(share=True, quiet=True)

---

## Step 6 — Score it, then break it

Write three questions your documents answer, and let RAGAS score them the way Lab 6 did: **faithfulness** (is every claim backed by the retrieved chunks?) and **answer relevancy** (does it answer what was asked?). The judge is `gpt-4o`; three questions cost a few cents.

Then do the part that makes a capstone good: find a question your assistant gets **wrong**, and work out which step caused it. That goes in your presentation.

In [ ]:
from ragas import evaluate, EvaluationDataset
from ragas.metrics import Faithfulness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

eval_questions = [                           # TODO: three real questions from your domain
    "YOUR EVAL QUESTION 1",
    "YOUR EVAL QUESTION 2",
    "YOUR EVAL QUESTION 3",
]

rows = []
for q in eval_questions:
    answer, docs, _ = rag(q)
    rows.append({"user_input": q, "response": answer, "retrieved_contexts": list(docs)})

judge = LangchainLLMWrapper(ChatOpenAI(model=JUDGE_MODEL, temperature=0))
results = evaluate(EvaluationDataset.from_list(rows), metrics=[Faithfulness(), AnswerRelevancy()], llm=judge)
results.to_pandas()[["user_input", "faithfulness", "answer_relevancy"]]

If the RAGAS cell errors (a package update, a quota), use the manual rubric below. It is the same review by hand: read the answer, read the sources, decide.

In [ ]:
for q in eval_questions:
    answer, docs, metas = rag(q)
    print("Q:", q)
    print("A:", answer[:250])
    print("Sources:", [m["source"] for m in metas])
    print("Grounded 1–5?  Relevant 1–5?  (you decide)\n")

---

## Submission checklist

- [ ] The knowledge base is a real domain, not the course example
- [ ] The system prompt is written for that domain and its users
- [ ] The app is live on a public link, with your example questions
- [ ] You have one question the system answers well and one it gets wrong, and you know which step caused the wrong one
- [ ] You can say what chunk size you chose and why
- [ ] RAGAS scores or the manual rubric for three questions

**Presentation (5 minutes):**

1. *"I built a [domain] assistant for [who]."* (30 seconds)
2. The pipeline: documents → chunks → Chroma → retrieval → `gpt-4o-mini` → Gradio. Say what you chose at each step. (60 seconds)
3. Live demo: two good answers and one failure. (2 minutes)
4. What surprised you, and what you would change. (90 seconds)

**Want a regression test?** Lab 12 shows how to turn your good and bad questions into a golden set you can rerun after every change.